# Mean-Only Estimation (1D)

Fits only the mean function using `n_components=0` on synthetic irregular 1D data.
The true mean is `μ(t) = t`. No components are estimated.

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import torch

from irregpca import IrregPCA, IrregPCAConfig

torch.manual_seed(42)

def true_mean(t: torch.Tensor) -> torch.Tensor:
    return t

In [ ]:
# Sample irregular observations (same setup as synthetic_irregular_1d.ipynb)
n_samples = 200
obs_per = 50
sigma = 0.05

ids_list, locs_list, vals_list = [], [], []
for i in range(n_samples):
    t = torch.rand(obs_per, 1)
    y = true_mean(t.squeeze()) + sigma * torch.randn(obs_per)
    ids_list.append(torch.full((obs_per,), float(i)))
    locs_list.append(t)
    vals_list.append(y)

sample_ids = torch.cat(ids_list)
locations  = torch.cat(locs_list, dim=0)
values     = torch.cat(vals_list)

In [ ]:
# Fit mean only (n_components=0)
cfg = IrregPCAConfig(
    n_components=0,
    epochs=600,
    lr=0.001,
    patience=300,
    random_state=42,
    verbose=True,
    hidden_width=30,
    hidden_depth=2,
)
result = IrregPCA(config=cfg).fit(
    sample_ids=sample_ids, locations=locations, values=values
)

In [ ]:
# Evaluate on a dense grid
grid = torch.linspace(0, 1, 500).unsqueeze(-1)
mu_hat  = result.mean(grid).cpu()
mu_true = true_mean(grid.squeeze())

mse = ((mu_hat - mu_true) ** 2).mean()
print(f"Mean MSE on grid: {mse:.6f}")
print(f"Best epoch: {result.history.best_epochs[0]}")
print(f"n_components: {result.n_components}")

In [ ]:
# Plot fitted vs true mean
t = grid.squeeze().cpu().numpy()

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(t, mu_true.numpy(), lw=2, color="gray", linestyle="--", label="true")
ax.plot(t, mu_hat.numpy(), lw=2, label="fitted")
ax.set_title("Mean-only fit:  μ(t) = t")
ax.set_xlabel("t")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("mean_only_1d.png", dpi=150)
plt.show()